# Dashboard #4 — Feature Engineering

Builds the final feature set for the multivariate model, based on `02_indicator_screening`
results plus horizon selection.

**Horizon: h=6.** Chosen as a middle ground — h=3 behaves closer to a nowcast with limited
lead time; h=12 showed weaker signal across nearly all indicators in screening and further
dilutes the target (more months get labeled positive purely by being anywhere within a wider
window, blurring the distinction between "imminent" and "still 11 months out").

**Initial feature set (6):** `BAA10Y`, `CFNAI`, `ICSA`, `is_inverted`, `PAYEMS`, `INDPRO`.
Selected from screening leaders plus sector coverage (credit, broad activity, labor,
manufacturing) rather than a strict top-N by PR-AUC alone.

`is_inverted = (T10Y2Y < 0)`: a binarized transform of the yield-curve spread, replacing the
raw level that performed weakly in screening — testing whether the *inversion event* itself
carries signal that the raw value did not.

Note: this feature set is provisional — multicollinearity is checked in
`04_model_training.ipynb`, which led to dropping `PAYEMS` and later `is_inverted`.

In [4]:
import pandas as pd

monthly_1990 = pd.read_csv('../data/processed/monthly_1990.csv', index_col=0, parse_dates=True)
target_h6 = pd.read_csv('../data/processed/target_h6.csv', index_col=0, parse_dates=True).squeeze()
monthly_1990['is_inverted'] = (monthly_1990['T10Y2Y'] < 0).astype(int)

feature_cols = ['BAA10Y', 'CFNAI', 'ICSA', 'is_inverted', 'PAYEMS', 'INDPRO']

combined = pd.concat(
    [monthly_1990[feature_cols], target_h6],
    axis=1
).dropna()

X = combined[feature_cols]
y = combined[target_h6.name]

X.shape, y.shape, y.mean()

((433, 6), (433,), np.float64(0.24942263279445728))

## Save combined feature/target table for model training

In [5]:
combined.to_csv('../data/processed/model_input_h6.csv')